# PJM API Historical Data Backfill

In [ ]:
import pandas as pd
import gridstatus as gs
import os

In [ ]:
db = os.environ['TIMESCALEDB_URL']
eia_api = os.environ["EIA_API_KEY"]
pjm_api_key = os.environ["PJM_API_KEY"]

In [ ]:
import logging
logging.getLogger("gridstatus").setLevel(logging.WARNING)  # avoid API key ending up in committed cell output

pjm = gs.PJM(api_key=pjm_api_key, retries=6)  # default of 3 isn't enough backoff to outlast PJM's rate limiting on paginated pulls

In [ ]:
load_test = pjm.get_load_metered_hourly("2023-01-01", "2023-01-08")
load_test['Zone'].unique()

In [ ]:
zones = pd.read_csv('../data/processed/pjm_weather_zones.csv')
zone_ids = ["RTO"] + zones['zone_id'].tolist()

# get_load_metered_hourly doesn't auto-chunk date ranges (unlike get_lmp) and
# PJM's API rejects a single multi-year request, so chunk by calendar year.
end_date = pd.Timestamp("2026-08-07")
year_starts = pd.date_range("2023-01-01", end_date, freq="YS")

load_frames = []
for start in year_starts:
    end = min(start + pd.DateOffset(years=1), end_date)
    load_frames.append(
        pjm.get_load_metered_hourly(start.strftime("%Y-%m-%d"), end.strftime("%Y-%m-%d"))
    )

load_hourly = pd.concat(load_frames, ignore_index=True)
load_hourly = load_hourly[load_hourly['Zone'].isin(zone_ids)].reset_index(drop=True)

# schema.md's `time` column is interval start; interval end is always +1h for this
# hourly feed, so it's redundant to store.
load_hourly = load_hourly.rename(columns={"Interval Start": "time"}).drop(columns=["Interval End"])
load_hourly

In [ ]:
load_hourly.to_csv("../data/interim/pjm_load_hourly_2023_present.csv", index=False)

### EIA-930 RTO-level enrichment (demand forecast, net generation, interchange)

Not redundant with the PJM zone-level pull above — these fields don't exist in PJM's
`hrl_load_metered` feed at all, and EIA only reports them at whole-RTO granularity,
not per-zone. Lands as a single `subregion IS NULL` row per hour in the `load` table.

In [ ]:
eia = gs.EIA(api_key=eia_api)

# get_grid_monitor is RTO-level only (no per-zone breakdown) and can't be filtered by
# date - it fetches all available history for the area in one call.
eia_grid_monitor = eia.get_grid_monitor(area_id="PJM")
eia_grid_monitor[["Interval Start", "Demand", "Demand Forecast", "Net Generation", "Total Interchange"]].head()

In [ ]:
# Shape to the `load` table's schema: one RTO-total row per hour, subregion left
# NULL to match schema.md's "null for BA-level totals" convention - these values
# are RTO-wide, not per-zone, so they don't belong on the zone rows from the PJM pull.
eia_rto = eia_grid_monitor.rename(columns={
    "Interval Start": "time",
    "Demand": "demand_mw",
    "Demand Forecast": "demand_forecast_mw",
    "Net Generation": "net_generation_mw",
    "Total Interchange": "total_interchange_mw",
})
eia_rto["ba_code"] = "PJM"
eia_rto["subregion"] = None

eia_rto = eia_rto[["time", "ba_code", "subregion", "demand_mw", "demand_forecast_mw", "net_generation_mw", "total_interchange_mw"]]
eia_rto = eia_rto[eia_rto["time"] >= "2023-01-01"].reset_index(drop=True)
eia_rto

In [ ]:
eia_rto.to_csv("../data/interim/eia_rto_hourly_2023_present.csv", index=False)